In [9]:
import gmsh
import sys
from collections import defaultdict
gmsh.initialize()

In [10]:
# ═════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═════════════════════════════════════════════════════════════════════

FILES = [
    ("upper",    "skin.stp"),      # skin exports upper + lower as one file
    ("rib0",     "rib0.stp"),
    ("rib750",   "rib750.stp"),
    ("rib1500",  "rib1500.stp"),
    ("rib2250",  "rib2250.stp"),
    ("rib3000",  "rib3000.stp"),
    ("mainspar", "spar_main.stp"),
    ("tailspar", "spar_tail.stp"),
]

# Surface physical group tags — match wing.geo exactly
# skin.stp contains both upper (14) and lower (15) surfaces;
# they are split by the key "upper"/"lower" after fragment via
# bounding-box z-sign (see Step 6).
SURF_TAGS = {
    "upper"   : 14,
    "lower"   : 15,
    "rib0"    : 38,
    "rib750"  : 39,
    "rib1500" : 40,
    "rib2250" : 41,
    "rib3000" : 42,
    "mainspar": 43,
    "tailspar": 44,
}

# Rib perimeter curve physical group tags — match wing.geo exactly
RIB_CURVE_TAGS = {
    "rib0"   : 45,
    "rib750" : 46,
    "rib1500": 47,
    "rib2250": 48,
    "rib3000": 49,
}
SPAR_CURVE_TAGS = {
    "mainspar": 50,
    "tailspar": 51,
}

# Root BC — match wing.geo Physical Curve("root", 52)
TAG_ROOT = 52

# Mesh
MESH_SIZE  = 10.0   # mm
MESH_ALGO  = 6      # Frontal-Delaunay
OUTPUT_MSH = "wing_full.msh"

# OCC tolerance — larger than CATIA float noise, smaller than mesh size
OCC_TOL = 1e-2   # mm

# Tolerance for bounding-box curve detection
BC_TOL = 1.0     # mm


# ═════════════════════════════════════════════════════════════════════
# STEP 1 — Import
# ═════════════════════════════════════════════════════════════════════
gmsh.model.add("wing")

gmsh.option.setNumber("Geometry.Tolerance",        OCC_TOL)
gmsh.option.setNumber("Geometry.ToleranceBoolean", OCC_TOL)

print("\n" + "="*60)
print("STEP 1 — Import .stp files")
print("="*60)

# tag_map: component name -> set of OCC surface tags BEFORE fragment
# For skin.stp we use the key "upper" provisionally; the upper/lower
# split is resolved in Step 6 after we can inspect geometry.
tag_map = {}

for name, filepath in FILES:
    tags_before = {t for (_, t) in gmsh.model.getEntities(2)}
    try:
        gmsh.model.occ.importShapes(filepath)
        gmsh.model.occ.synchronize()
    except Exception as e:
        print(f"  FAIL  {filepath} -> {e}")
        gmsh.finalize()
        sys.exit(1)
    tags_after = {t for (_, t) in gmsh.model.getEntities(2)}
    new_tags   = tags_after - tags_before
    tag_map[name] = new_tags
    print(f"  ok  {name:10s}  ({filepath})  ->  geo tags {sorted(new_tags)}")

# skin.stp contains both surfaces; register "lower" as empty for now
# — it will be filled during the upper/lower split in Step 6
tag_map["lower"] = set()


# ═════════════════════════════════════════════════════════════════════
# STEP 2 — Fragment
#
# fragment() detects geometrically coincident OCC entities across the
# independently imported .stp bodies and merges them into shared
# topological edges.  It returns a mapping from old surface tags to
# new surface tags which is the ONLY reliable way to update tag_map.
#
# removeAllDuplicates() is NOT called — it renumbers tags without
# returning a mapping, silently emptying tag_map entries.
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2 — Fragment")
print("="*60)

all_surfs_before = gmsh.model.getEntities(dim=2)
n_edges_before   = len(gmsh.model.getEntities(dim=1))

result, mapping = gmsh.model.occ.fragment(all_surfs_before, [])
gmsh.model.occ.synchronize()

n_surfs_after = len(gmsh.model.getEntities(dim=2))
n_edges_after = len(gmsh.model.getEntities(dim=1))

print(f"  Surfaces : {len(all_surfs_before)} -> {n_surfs_after}")
print(f"  Edges    : {n_edges_before} -> {n_edges_after}"
      f"  (delta = {n_edges_after - n_edges_before:+d})")

# Update tag_map via fragment mapping
old_to_new = {}
for i, (dim, old_tag) in enumerate(all_surfs_before):
    old_to_new[old_tag] = [t for (d, t) in mapping[i] if d == 2]

for name in tag_map:
    updated = set()
    for old_tag in tag_map[name]:
        updated.update(old_to_new.get(old_tag, [old_tag]))
    tag_map[name] = updated

print()
for name, tags in tag_map.items():
    print(f"  {name:10s} -> {sorted(tags)}")

missing = [n for n, tags in tag_map.items()
           if not tags and n != "lower"]   # lower is filled later
if missing:
    print(f"\n  ERROR: empty tag_map after fragment for: {missing}")
    gmsh.finalize()
    sys.exit(1)


# ═════════════════════════════════════════════════════════════════════
# STEP 3 — Remove ALL stale physical groups
#
# Merge imports not only geometry but also any Physical Groups that
# were defined in the .stp/geo files.  Those groups reference
# pre-fragment curve/surface tags which are now stale.
# We purge everything and rebuild from scratch in Steps 5-7.
#
# removePhysicalGroups([]) with an empty list removes ALL groups
# across ALL dimensions in one call — the only reliable approach.
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 3 — Remove stale physical groups")
print("="*60)

before_surf  = gmsh.model.getPhysicalGroups(2)
before_curve = gmsh.model.getPhysicalGroups(1)

gmsh.model.removePhysicalGroups([])   # [] = all dims, all tags

after_surf  = gmsh.model.getPhysicalGroups(2)
after_curve = gmsh.model.getPhysicalGroups(1)

print(f"  Surface groups : {len(before_surf)} -> {len(after_surf)}")
print(f"  Curve  groups  : {len(before_curve)} -> {len(after_curve)}")
print(f"  ok  All stale physical groups removed")


# ═════════════════════════════════════════════════════════════════════
# STEP 4 — Connectivity check
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 4 — Connectivity check")
print("="*60)

edge_to_surfaces = defaultdict(list)
for (dim, edge_tag) in gmsh.model.getEntities(1):
    upward, _ = gmsh.model.getAdjacencies(dim, edge_tag)
    for surf_tag in upward:
        edge_to_surfaces[edge_tag].append(surf_tag)

shared_edges = [(e, s) for e, s in edge_to_surfaces.items() if len(s) >= 2]
print(f"  Shared edges (>= 2 surfaces) : {len(shared_edges)}")

if not shared_edges:
    print("  ERROR: no shared edges — fragment did not connect components.")
    print("  Increase OCC_TOL or check that .stp boundaries coincide.")
    gmsh.finalize()
    sys.exit(1)

# DFS
surface_graph = defaultdict(set)
for edge, surfs in shared_edges:
    for i in range(len(surfs)):
        for j in range(i + 1, len(surfs)):
            surface_graph[surfs[i]].add(surfs[j])
            surface_graph[surfs[j]].add(surfs[i])

all_surf_tags = [t for (_, t) in gmsh.model.getEntities(2)]
visited, n_components = set(), 0
for s in all_surf_tags:
    if s not in visited:
        n_components += 1
        stack = [s]
        while stack:
            cur = stack.pop()
            if cur not in visited:
                visited.add(cur)
                stack.extend(surface_graph[cur] - visited)

print(f"  Connected components         : {n_components}")
if n_components != 1:
    print("  ERROR: geometry is not one connected shell.")
    gmsh.finalize()
    sys.exit(1)
print("  ok  ONE connected shell")


# ═════════════════════════════════════════════════════════════════════
# STEP 5 — Upper / lower skin split
#
# skin.stp exports upper and lower surfaces together under the key
# "upper" in tag_map.  After fragment, the skin is split at each rib
# and spar station into sub-panels.  We separate upper (z centroid > 0)
# from lower (z centroid < 0) by inspecting each surface's bounding box.
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 5 — Split upper / lower skin")
print("="*60)

upper_tags = set()
lower_tags = set()

for t in tag_map["upper"]:
    xmn, ymn, zmn, xmx, ymx, zmx = gmsh.model.getBoundingBox(2, t)
    z_mid = (zmn + zmx) / 2.0
    if z_mid >= 0:
        upper_tags.add(t)
    else:
        lower_tags.add(t)

tag_map["upper"] = upper_tags
tag_map["lower"] = lower_tags

print(f"  upper : {len(upper_tags)} surface(s)  {sorted(upper_tags)}")
print(f"  lower : {len(lower_tags)} surface(s)  {sorted(lower_tags)}")

if not upper_tags or not lower_tags:
    print("  WARN: could not separate upper/lower skin.")
    print("  Check that the skin surface spans both positive and negative z.")


# ═════════════════════════════════════════════════════════════════════
# STEP 6 — Physical Groups: surfaces (material tags)
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 6 — Physical Groups (surfaces)")
print("="*60)

for name, phys_tag in SURF_TAGS.items():
    geo_tags = sorted(tag_map.get(name, []))
    if geo_tags:
        gmsh.model.addPhysicalGroup(2, geo_tags, tag=phys_tag, name=name)
        print(f"  ok  {name:10s}  phys={phys_tag}  ->  {len(geo_tags)} surface(s)  {geo_tags}")
    else:
        print(f"  WARN  {name:10s}  phys={phys_tag}  ->  no surfaces found")


# ═════════════════════════════════════════════════════════════════════
# STEP 7 — Physical Groups: curves (BC)
#
# curves_at_y finds post-fragment curve tags geometrically.
#
# For root (y=0): exclude rib inner hole boundaries.
#   A rib inner hole edge is adjacent to ONLY rib surfaces (rib_surf_tags).
#   A structural boundary edge (skin outer, spar root, rib perimeter)
#   is adjacent to at least one non-rib surface, OR is the rib perimeter
#   shared with the skin (upward contains a skin surface tag).
#   We exclude any curve whose upward neighbours are ALL rib surfaces.
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 7 — Physical Groups (curves)")
print("="*60)

# All rib geo surface tags (used to detect inner hole edges)
rib_surf_tags = set()
for name in ["rib0", "rib750", "rib1500", "rib2250", "rib3000"]:
    rib_surf_tags.update(tag_map.get(name, set()))

# def curves_at_y(y_target, tol=BC_TOL):
#     """
#     Return post-fragment curve tags at y ~ y_target that belong to the
#     structural attachment boundary.

#     Excluded:
#       - Rib inner hole boundaries: upward neighbours are ALL rib surfaces.

#     Included:
#       - Skin outer boundary:     upward = {skin surface}
#       - Spar root edges:         upward = {spar surface}
#       - Rib outer perimeter:     upward = {rib surface, skin surface}
#                                  (not all ribs → included)
#     """
#     found = []
#     for (dim, tag) in gmsh.model.getEntities(1):
#         _, ymn, _, _, ymx, _ = gmsh.model.getBoundingBox(1, tag)
#         if abs(ymn - y_target) < tol and abs(ymx - y_target) < tol:
#             upward, _ = gmsh.model.getAdjacencies(dim, tag)
#             # Skip if every adjacent surface is a rib — inner hole boundary
#             if set(upward).issubset(rib_surf_tags):
#                 continue
#             found.append(tag)
#     return found

# REPLACE the entire curves_at_y function and the root_curves block with this:

def skin_boundary_curves_at_y(y_target, tol=BC_TOL):
    """
    Find root BC curves by walking the boundary of skin surfaces.
    More robust than filtering all curves because it is grounded in
    topology: only curves that are actually on the skin boundary at
    y ~ y_target are returned. Rib interior holes are never on the
    skin boundary, so no exclusion logic is needed.
    """
    skin_surf_tags = tag_map.get("upper", set()) | tag_map.get("lower", set())
    found = set()
    for surf_tag in skin_surf_tags:
        bounds = gmsh.model.getBoundary(
            [(2, surf_tag)], oriented=False, combined=False
        )
        for (dim, curve_tag) in bounds:
            if dim != 1:
                continue
            _, ymn, _, _, ymx, _ = gmsh.model.getBoundingBox(1, curve_tag)
            if abs(ymn - y_target) < tol and abs(ymx - y_target) < tol:
                found.add(curve_tag)
    return sorted(found)


# ```

# ---

# After these two changes, expected output:
# ```
# CHECK 2
#   ok    rib0         217 boundary nodes shared with skin (same tag)
#   ...

# CHECK 1
#   ok    phys curve 'root' tag=52  N curves  M nodes

# RESULT :  ALL CHECKS PASSED


def curves_of_component_at_y(component_names, y_target, tol=BC_TOL):
    """
    Return curves that belong to the given components AND are at y ~ y_target.
    Used to assign rib perimeter curve groups (ribrootc, rib750c, etc.).
    """
    comp_surfs = set()
    for name in component_names:
        comp_surfs.update(tag_map.get(name, set()))

    found = []
    for (dim, tag) in gmsh.model.getEntities(1):
        _, ymn, _, _, ymx, _ = gmsh.model.getBoundingBox(1, tag)
        if abs(ymn - y_target) < tol and abs(ymx - y_target) < tol:
            upward, _ = gmsh.model.getAdjacencies(dim, tag)
            # Curve must be adjacent to this component's surfaces
            if set(upward) & comp_surfs:
                found.append(tag)
    return found

# # FROM GPT
# def root_curves_structural(y_target, tol=BC_TOL):
#     # surfaces sets
#     skin_surfs = tag_map["upper"] | tag_map["lower"]
#     spar_surfs = tag_map["mainspar"] | tag_map["tailspar"]
#     rib_surfs  = set()
#     for r in ["rib0","rib750","rib1500","rib2250","rib3000"]:
#         rib_surfs |= tag_map.get(r, set())

#     found = []
#     for (dim, e) in gmsh.model.getEntities(1):
#         _, ymn, _, _, ymx, _ = gmsh.model.getBoundingBox(1, e)
#         if abs(ymn - y_target) > tol or abs(ymx - y_target) > tol:
#             continue

#         upward, _ = gmsh.model.getAdjacencies(1, e)
#         U = set(upward)

#         # (A) boundary edges of skin/spar at root -> always keep
#         if len(U) == 1 and (U & (skin_surfs | spar_surfs)):
#             found.append(e)
#             continue

#         # (B) shared edges at root you still want clamped:
#         # rib perimeter shared with skin/spar (NOT rib-hole)
#         if (U & rib_surfs) and (U & (skin_surfs | spar_surfs)):
#             found.append(e)
#             continue

#         # otherwise: skip (includes rib-hole loops and interior edges)
#     return found


# Root BC (tag 52) — structural perimeter at y=0
# root_curves = curves_at_y(0.0)

# Root BC (tag 52)
# root_curves = skin_boundary_curves_at_y(0.0)
# if root_curves:
#     gmsh.model.addPhysicalGroup(1, root_curves, tag=TAG_ROOT, name="root")
#     print(f"  ok  root       phys={TAG_ROOT}  ->  {len(root_curves)} curves")
# else:
#     print(f"  WARN  root not found  (BC_TOL={BC_TOL} mm)")
#     print(f"        skin surfaces at y=0: check that upper/lower split is correct")

# Replace in STEP 7:

root_curves = curves_of_component_at_y(
    ["upper", "lower", "rib0", "mainspar", "tailspar"], 0.0
)
print(f"  [DEBUG] root_curves before filter : {len(root_curves)}")
root_curves = [
    t for t in root_curves
    if not set(gmsh.model.getAdjacencies(1, t)[0]).issubset(rib_surf_tags)
]

# ADD THIS:
if root_curves:
    gmsh.model.addPhysicalGroup(1, root_curves, tag=TAG_ROOT, name="root")
    print(f"  ok  root       phys={TAG_ROOT}  ->  {len(root_curves)} curves")
else:
    print(f"  WARN  root not found at y=0  (BC_TOL={BC_TOL} mm)")
    print(f"        Try increasing BC_TOL or check .stp geometry at root")



# # FROM GPT
# if root_curves:
#     gmsh.model.addPhysicalGroup(1, root_curves, tag=TAG_ROOT, name="root")
#     print(f"  ok  root       phys={TAG_ROOT}  ->  {len(root_curves)} curves")
# else:
#     print(f"  WARN  root not found  (BC_TOL={BC_TOL} mm)")

# Rib perimeter curves (ribrootc=45 ... ribtailc=49)
RIB_Y = {
    "rib0"   : (0.0,    45),
    "rib750" : (750.0,  46),
    "rib1500": (1500.0, 47),
    "rib2250": (2250.0, 48),
    "rib3000": (3000.0, 49),
}
for rib_name, (y_pos, curve_tag) in RIB_Y.items():
    skin_names = ["upper", "lower"]
    spar_names = ["mainspar", "tailspar"]
    # Rib perimeter = curves at this y shared between rib and skin/spar
    rib_curves = curves_of_component_at_y(
        [rib_name] + skin_names + spar_names, y_pos
    )
    # Exclude inner hole: remove curves adjacent only to rib surfaces
    rib_curves = [
        t for t in rib_curves
        if not set(gmsh.model.getAdjacencies(1, t)[0]).issubset(rib_surf_tags)
    ]
    if rib_curves:
        gmsh.model.addPhysicalGroup(1, rib_curves, tag=curve_tag,
                                    name=f"{rib_name}c")
        print(f"  ok  {rib_name:10s}c  phys={curve_tag}  "
              f"->  {len(rib_curves)} curves")
    else:
        print(f"  WARN  {rib_name}c not found at y={y_pos}")

# Spar curves (mainsparc=50, tailsparc=51)
for spar_name, curve_tag in SPAR_CURVE_TAGS.items():
    spar_surfs = tag_map.get(spar_name, set())
    spar_curves = []
    for (dim, tag) in gmsh.model.getEntities(1):
        upward, _ = gmsh.model.getAdjacencies(dim, tag)
        if set(upward) & spar_surfs:
            spar_curves.append(tag)
    if spar_curves:
        gmsh.model.addPhysicalGroup(1, spar_curves, tag=curve_tag,
                                    name=f"{spar_name}c")
        print(f"  ok  {spar_name:10s}c  phys={curve_tag}  "
              f"->  {len(spar_curves)} curves")
    else:
        print(f"  WARN  {spar_name}c not found")


# ═════════════════════════════════════════════════════════════════════
# STEP 8 — Mesh and export
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 8 — Mesh")
print("="*60)

gmsh.option.setNumber("Mesh.CharacteristicLengthMax",           MESH_SIZE)
gmsh.option.setNumber("Mesh.CharacteristicLengthMin",           MESH_SIZE / 4)
gmsh.option.setNumber("Mesh.CharacteristicLengthFromCurvature", 1)
gmsh.option.setNumber("Mesh.MinimumCirclePoints",               20)
gmsh.option.setNumber("Mesh.Algorithm",                         MESH_ALGO)

gmsh.model.mesh.generate(2)

n_nodes = len(gmsh.model.mesh.getNodes()[0])
print(f"  Nodes : {n_nodes}")

gmsh.write(OUTPUT_MSH)
print(f"  Saved -> {OUTPUT_MSH}")


# ═════════════════════════════════════════════════════════════════════
# SUMMARY
# ═════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"  Output       : {OUTPUT_MSH}")
print(f"  Nodes        : {n_nodes}")
print(f"  Shared edges : {len(shared_edges)}")
print(f"  Components   : {n_components}")
print("""
  Physical group tags for wing.py:

    Surfaces (cell_tags):
      14 -> upper skin    39 -> rib750     43 -> mainspar
      15 -> lower skin    40 -> rib1500    44 -> tailspar
      38 -> rib0          41 -> rib2250
                          42 -> rib3000

    Curves (facet_tags):
      52 -> root BC (clamped)

  Next steps:
      python check_mesh.py      verify conformity
      python wing.py            FEM solver
""")

# gmsh.finalize()



STEP 1 — Import .stp files
Info    :  - Label 'Shapes/Part1/=>[0:1:1:2]/COMPOUND' (3D)
Info    :  - Label 'Shapes/Part1/=>[0:1:1:3]/SHELL' (2D)
Info    :  - Color (0.952941, 0.996078, 0.694118) (2D & Surfaces)
Info    :  - Label 'Shapes/COMPOUND' (1D)
Info    :  - Color (1, 1, 1) (1D & Curves)
Info    :  - Label 'Shapes/COMPOUND' (1D)
Info    :  - Color (1, 1, 1) (1D & Curves)
Info    :  - Label 'Shapes/SHELL' (1D)
Info    :  - Label 'Shapes/SHELL' (1D)
Info    :  - Label 'Shapes/SHELL' (1D)
Info    :  - Label 'Shapes/SHELL' (1D)
Info    :  - Label 'Shapes/SHELL' (1D)
Info    :  - Label 'Shapes/SHELL' (1D)
  ok  upper       (skin.stp)  ->  geo tags [1, 2]
Info    :  - Label 'Shapes/Open CASCADE STEP translator 7.9 1/1/Open CASCADE STEP translator 7.9 1.1' (2D)
Info    :  - Label 'Shapes/Open CASCADE STEP translator 7.9 1/2/Open CASCADE STEP translator 7.9 1.2' (3D)
  ok  rib0        (rib0.stp)  ->  geo tags [3]
Info    :  - Label 'Shapes/Open CASCADE STEP translator 7.9 1/1/Open CASCA

In [11]:
import meshio
import numpy as np

# Save a mesh containing ONLY the root curves (physical group 52)
gmsh.model.mesh.generate(1)   # ensure 1D elements are meshed on curves

# Remove all physical groups except root
all_phys = gmsh.model.getPhysicalGroups()
for (dim, tag) in all_phys:
    if not (dim == 1 and tag == TAG_ROOT):
        gmsh.model.removePhysicalGroups([(dim, tag)])

gmsh.write("root_curves.msh")
print("  Saved root curves -> root_curves.msh")

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 10%] Meshing curve 2 (BSpline)
Info    : [ 10%] Meshing curve 3 (BSpline)
Info    : [ 10%] Meshing curve 4 (BSpline)
Info    : [ 10%] Meshing curve 5 (BSpline)
Info    : [ 10%] Meshing curve 6 (BSpline)
Info    : [ 10%] Meshing curve 7 (BSpline)
Info    : [ 10%] Meshing curve 8 (BSpline)
Info    : [ 20%] Meshing curve 9 (BSpline)
Info    : [ 20%] Meshing curve 10 (BSpline)
Info    : [ 20%] Meshing curve 11 (BSpline)
Info    : [ 20%] Meshing curve 12 (BSpline)
Info    : [ 20%] Meshing curve 13 (BSpline)
Info    : [ 20%] Meshing curve 14 (BSpline)
Info    : [ 20%] Meshing curve 15 (BSpline)
Info    : [ 20%] Meshing curve 16 (BSpline)
Info    : [ 30%] Meshing curve 17 (BSpline)
Info    : [ 30%] Meshing curve 18 (BSpline)
Info    : [ 30%] Meshing curve 19 (BSpline)
Info    : [ 30%] Meshing curve 20 (BSpline)
Info    : [ 30%] Meshing curve 21 (BSpline)
Info    : [ 30%] Meshing curve 22 (BSpline)
Info    : [ 30%] 

In [12]:
root_curves = curves_of_component_at_y(
    ["upper", "lower", "rib0", "mainspar", "tailspar"], 0.0
)
print(f"\n  [DEBUG] Root curves bounding boxes:")
for t in root_curves:
    xmn, ymn, zmn, xmx, ymx, zmx = gmsh.model.getBoundingBox(1, t)
    upward, _ = gmsh.model.getAdjacencies(1, t)
    print(f"    curve {t:4d}  y=[{ymn:.2f}, {ymx:.2f}]  adjacent_surfs={list(upward)}")


  [DEBUG] Root curves bounding boxes:
    curve   24  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(9), np.int32(25)]
    curve   28  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(11), np.int32(26)]
    curve   30  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(12), np.int32(27)]
    curve   49  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(21), np.int32(27)]
    curve   52  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(23), np.int32(26)]
    curve   54  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(24), np.int32(25)]
    curve   55  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(25), np.int32(26), np.int32(47)]
    curve   56  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(26), np.int32(27), np.int32(43)]
    curve   57  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(26)]
    curve   58  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(26)]
    curve   59  y=[-0.00, 0.00]  adjacent_surfs=[np.int32(27)]


In [13]:
import meshio
import numpy as np

# Save a mesh containing ONLY the root curves (physical group 52)
gmsh.model.mesh.generate(1)   # ensure 1D elements are meshed on curves

root_curves = [
    t for t in root_curves
    if not (
        len(set(gmsh.model.getAdjacencies(1, t)[0])) >= 2  # only filter if 2+ adjacent surfs
        and set(gmsh.model.getAdjacencies(1, t)[0]).issubset(rib_surf_tags)
    )
]

# Remove all physical groups except root
all_phys = gmsh.model.getPhysicalGroups()
for (dim, tag) in all_phys:
    if not (dim == 1 and tag == TAG_ROOT):
        gmsh.model.removePhysicalGroups([(dim, tag)])

gmsh.write("root_curves.msh")
print("  Saved root curves -> root_curves.msh")

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (BSpline)
Info    : [ 10%] Meshing curve 2 (BSpline)
Info    : [ 10%] Meshing curve 3 (BSpline)
Info    : [ 10%] Meshing curve 4 (BSpline)
Info    : [ 10%] Meshing curve 5 (BSpline)
Info    : [ 10%] Meshing curve 6 (BSpline)
Info    : [ 10%] Meshing curve 7 (BSpline)
Info    : [ 10%] Meshing curve 8 (BSpline)
Info    : [ 20%] Meshing curve 9 (BSpline)
Info    : [ 20%] Meshing curve 10 (BSpline)
Info    : [ 20%] Meshing curve 11 (BSpline)
Info    : [ 20%] Meshing curve 12 (BSpline)
Info    : [ 20%] Meshing curve 13 (BSpline)
Info    : [ 20%] Meshing curve 14 (BSpline)
Info    : [ 20%] Meshing curve 15 (BSpline)
Info    : [ 20%] Meshing curve 16 (BSpline)
Info    : [ 30%] Meshing curve 17 (BSpline)
Info    : [ 30%] Meshing curve 18 (BSpline)
Info    : [ 30%] Meshing curve 19 (BSpline)
Info    : [ 30%] Meshing curve 20 (BSpline)
Info    : [ 30%] Meshing curve 21 (BSpline)
Info    : [ 30%] Meshing curve 22 (BSpline)
Info    : [ 30%] 